# Reproducing Senzai & Buzsaki (2017) from DANDI:000003

Paper. Senzai Y, Buzsaki G (2017). Physiological Properties and Behavioral Correlates of Hippocampal Granule Cells and Mossy Cells. *Neuron* 93, 691-704. doi:10.1016/j.neuron.2016.12.011.

Dataset. DANDI:000003, version 0.261122.2143 (doi:10.48324/dandi.000003/0.261122.2143). This version was asked for in the attempt's prompt; it is also the most recent published version, and the paper cites no version (its data statement names CRCNS). Every file is streamed from the archive; nothing is downloaded to disk.

Text read. The claims file (`claims/senzai-2017.yaml`) was written from the accepted author manuscript in PubMed Central (PMC5293146): Results, figure legends and STAR Methods. The author of this attempt read the same text under `paper/`.

What the dataset holds. 101 NWB files from 16 mice. YutaMouse57 (8 files) is not in the paper (dataset description) and is left out of every analysis. Each file has the sorted units with the authors' cell-type labels, spike times and stored waveforms (one file has none), the 1250 Hz LFP, brain states, dentate spikes, light pulses, the authors' tables of monosynaptic and zero-lag pairs (in most files), and T-maze trials and positions (42 files) or open-field positions (43 files). 17 files also hold the 20 kHz wide-band signal.

Claims, and which are tested (outcomes in the Summary at the end):

| Claim | Status |
|---|---|
| unit-counts | tested |
| first-stage-labels | tested |
| interneuron-rates | tested |
| gc-narrower-waveform | tested |
| optotag-drd2 | tested |
| mc-more-active | tested |
| gc-nrem-highest | tested |
| ca3-wake-highest | tested |
| nrem-wake-ratio | tested |
| theta-modulation | tested |
| gamma-modulation | tested |
| gamma-rate-correlation | tested |
| ds2-recruitment | tested |
| monosyn-gc-to-mc | tested |
| burst-facilitation | tested |
| state-transmission | tested |
| zero-lag-synchrony | tested |
| tmaze-central-active | tested |
| tmaze-multiple-fields | tested |
| tmaze-peak-info | tested |
| tmaze-splitters | tested |
| remap-positive | tested |
| remap-gc-lt-mc | tested |
| remap-fractions | tested |
| pv-remap | tested |
| inheritance-tmaze | tested |
| inheritance-openfield | tested |
| mc-deeper | tested |
| gc-mc-classification | tested |
| gc-mc-distance | blocked: metadata insufficient (no site coordinates) |
| gc-backpropagation | blocked: metadata insufficient (no site coordinates) |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time, zlib, resource
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
from scipy import stats
import senzai as S

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
PAPER = "senzai-2017"


def seed(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())


def mwu(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    return stats.mannwhitneyu(a, b, alternative="two-sided").pvalue


def kw(*g):
    g = [np.asarray(x, float)[np.isfinite(np.asarray(x, float))] for x in g]
    return stats.kruskal(*g).pvalue

In [ ]:
# Stream every file and compute the per-unit and per-pair measures (senzai.process_file),
# four files at a time. This is the long step (about 25 minutes).
t0 = time.time()
assets = S.list_assets()
print(len(assets), "assets in DANDI:000003 version", S.VERSION)
with ProcessPoolExecutor(4) as ex:
    R = list(ex.map(S.process_file, sorted(assets, key=lambda a: -a["size"])))
R = sorted(R, key=lambda r: r["path"])
print(f"streamed and processed in {(time.time() - t0) / 60:.1f} min")

In [ ]:
units_all = pd.DataFrame([u for r in R for u in r["units"]])
files = pd.DataFrame([dict(path=r["path"], subject=r["subject"], genotype=r["genotype"], span_h=r["span"] / 3600,
                           n_units=len(r["units"]), wideband=r.get("wideband", False), waveforms=r.get("has_waveforms", False),
                           n_ds2=r["n_ds2"], pulses20=r.get("n_pulses20", 0), tmaze_runs=r.get("tmaze", {}).get("n_runs", 0),
                           openfield=",".join(r.get("openfield", {}).keys()),
                           authors_mono=None if r["authors_mono"] is None else len(r["authors_mono"]),
                           **{f"{k}_h": v / 3600 for k, v in r["state_dur"].items()}) for r in R])
U = units_all[units_all.subject != S.EXCLUDED_SUBJECT].copy()
U["key"] = U.path + "#" + U.unit.astype(str)
DG_I = U.type.isin(["nI", "wI"]) & (U.region == "DG")

In [ ]:
print("files:", len(files), " subjects:", files.subject.nunique(), " units in all files:", len(units_all))
print("left out:", S.EXCLUDED_SUBJECT, (files.subject == S.EXCLUDED_SUBJECT).sum(), "files,",
      (units_all.subject == S.EXCLUDED_SUBJECT).sum(), "units")
print("\nfiles by subject and genotype:")
print(files.groupby(["subject", "genotype"]).agg(files=("path", "size"), units=("n_units", "sum"),
                                                 hours=("span_h", "sum")).round(1).to_string())
print("\ncell-type labels by region (paper's mice):")
print(pd.crosstab(U.cell_type, U.region, margins=True).to_string())
print("\nunits with a global_id:", (U.global_id >= 0).sum(), " duplicated global_ids:",
      U[U.global_id >= 0].global_id.duplicated().sum())
print("\nfiles with wide-band:", files.wideband.sum(), " with stored waveforms:", files.waveforms.sum(),
      " with T-maze runs:", (files.tmaze_runs > 0).sum(), " with open fields:", (files.openfield != "").sum(),
      " without the authors' monosynaptic table:", files.authors_mono.isna().sum())
print("\nstate hours (all files):", files[["awake_h", "quiet_h", "nrem_h", "rem_h"]].sum().round(1).to_dict())
print("session span (h): min %.2f median %.2f max %.2f" % (files.span_h.min(), files.span_h.median(), files.span_h.max()))
print("\nrates (Hz) by type, spikes over span: ")
print(U.groupby("type").rate.describe().round(3).to_string())

In [ ]:
# Units recorded on the same day in two files ('b' sessions): separate recordings or the same one twice?
pairs_same_day = []
for r in R:
    name = r["path"].split("ses-")[1]
    if "b16" in name:
        mate = [q for q in R if q["path"].split("ses-")[1] == name.replace("b16", "-16")]
        if mate:
            q = mate[0]
            a = {u["n_spikes"] for u in r["units"]}
            b = {u["n_spikes"] for u in q["units"]}
            pairs_same_day.append(dict(b_file=name[:24], span_h=round(r["span"] / 3600, 2), units=len(r["units"]),
                                       other_span_h=round(q["span"] / 3600, 2), other_units=len(q["units"]),
                                       shared_spike_counts=len(a & b), n_ds2=r["n_ds2"], other_n_ds2=q["n_ds2"]))
print(pd.DataFrame(pairs_same_day).to_string())

In [ ]:
# The electrodes table's channel map against the wide-band signal: for GC and MC in the wide-band files,
# the channel of the largest stored waveform (through units.electrodes) against the channel where the
# mean spike is largest in the 300 Hz high-passed wide band.
wb = U[U.type.isin(["GC", "MC"]) & U.site_wb.notna() & (U.site_wb >= 0) & U.site_table.notna()]
same = (wb.site_wb == wb.site_table)
print("GC/MC units with both sites:", len(wb), " same channel:", same.sum(), f"({same.mean():.3f})")
d = (wb.ds2_amp_wb - wb.ds2_amp_table).abs()
print("|DS2 amplitude difference| between the two sites (uV): median %.1f, 90th pct %.1f" % (d.median(), d.quantile(0.9)))
print("wide-band spikes used per unit: median", wb.site_wb_n.median(), " min", wb.site_wb_n.min())

In [ ]:
# Clock checks. (1) DS2 times against spikes: the peri-DS2 peak ratio with the true DS2 times and with
# the DS2 times shifted by 0.5 s. (2) Light pulses against spikes: DRD2 mice's tagged units with true
# and shifted pulse times are shown in the optotag section. (3) Positions against spikes: spatial
# information with the spikes as recorded and shifted by 60 s, in the T-maze section.
for t in ["GC", "MC", "CA3", "nI", "wI"]:
    x = U[(U.type == t) & U.ds2_ratio.notna()]
    print(f"{t:4s} n={len(x):4d}  DS2 peak ratio median: true {x.ds2_ratio.median():.2f}, shifted 0.5 s {x.ds2_ratio_shift.median():.2f}")
per_file = U[U.type.isin(["GC", "MC"]) & U.ds2_ratio.notna()].groupby("path")[["ds2_ratio", "ds2_ratio_shift"]].median()
print("files where the GC/MC median ratio is higher with true than shifted times:",
      (per_file.ds2_ratio > per_file.ds2_ratio_shift).sum(), "of", len(per_file))
print(per_file[per_file.ds2_ratio <= per_file.ds2_ratio_shift].round(2).to_string())

## unit-counts

> The second-stage classification divided dentate excitatory neurons into 586 putative granule cells (GC) and 500 putative mossy cells (MC).

Counted from the units' `cell_type` labels, once per recording day (each unit has its own `global_id`; the probe was moved 70 um after each day).

In [ ]:
cnt = U.drop_duplicates("global_id").type.value_counts()
print("GC:", cnt["GC"], " MC:", cnt["MC"], " (paper 586 and 500, tolerance 10%)")
print("including YutaMouse57:", (units_all.cell_type == "granule cell").sum(), "GC,", (units_all.cell_type == "mossy cell").sum(), "MC")
print("\nby subject:")
print(pd.crosstab(U.subject, U.type)[["GC", "MC", "CA3", "nI", "wI"]].to_string())

**Result: reproduced.** The labels give 583 GC and 503 MC, once per recording day (no global_id repeats), within 10% of 586 and 500.

## first-stage-labels

> Narrow-waveform interneurons have a trough-to-peak latency below 0.425 ms, and among wide-waveform units the excitatory cells have a burst index above the threshold separating them from wide-waveform interneurons.

Trough-to-peak latency from the mean of the first 200 stored waveforms on the channel of largest amplitude, upsampled to 100 kHz; burst index from the autocorrelogram (Methods). Threshold 1.2 (Methods) and 1.8 (Figure 1A legend). The 0.425 ms threshold is the dentate one; CA3 units have their own (0.525 ms, Methods) and are shown apart.

In [ ]:
def stage1(df, thr, tp_thr=0.425):
    out = {}
    n = df[df.type == "nI"]
    out["nI below tp"] = (n.tp_ms < tp_thr).mean(), len(n)
    e = df[df.type.isin(["GC", "MC"])]
    out["GC+MC wide & BI>thr"] = ((e.tp_ms >= tp_thr) & (e.burst_index > thr)).mean(), len(e)
    w = df[df.type == "wI"]
    out["wI wide & BI<=thr"] = ((w.tp_ms >= tp_thr) & (w.burst_index <= thr)).mean(), len(w)
    return out

dg = U[U.tp_ms.notna() & ((U.region == "DG") | U.type.isin(["GC", "MC"]))]
rows = []
for thr in (1.2, 1.8):
    for k, (frac, n) in stage1(dg, thr).items():
        rows.append(dict(threshold=thr, group=k, fraction=round(frac, 3), n=n, meets_90=frac >= 0.9))
print(pd.DataFrame(rows).to_string())
print("\nparts of each criterion (DG):")
for t in ["nI", "wI", "GC", "MC"]:
    x = dg[dg.type == t]
    print(f"{t}: n={len(x)} tp<0.425: {(x.tp_ms < 0.425).mean():.3f}  BI>1.2: {(x.burst_index > 1.2).mean():.3f}  BI>1.8: {(x.burst_index > 1.8).mean():.3f}"
          f"  tp median {x.tp_ms.median():.3f} ms, BI median {x.burst_index.median():.2f}")
ca3i = U[U.tp_ms.notna() & (U.region == "CA3") & U.type.isin(["nI", "wI"])]
print("\nCA3 interneurons (own threshold 0.525 ms): nI below 0.525:", round((ca3i[ca3i.type == "nI"].tp_ms < 0.525).mean(), 3),
      "n =", (ca3i.type == "nI").sum(), "; wI at or above:", round((ca3i[ca3i.type == "wI"].tp_ms >= 0.525).mean(), 3), "n =", (ca3i.type == "wI").sum())
print("units without a waveform (file without waveforms):", U[U.type.isin(["GC", "MC", "nI", "wI"]) & U.tp_ms.isna()].shape[0])
print("peak at the end of the stored 1.6 ms:", U[U.type.isin(['GC','MC','nI','wI'])].tp_at_end.sum())

**Result: partially reproduced.** Under either threshold (1.2 or 1.8), 98.5% of GC and MC are wide with a burst index above it, and 90.6% (1.2) or 92.6% (1.8) of dentate wide-waveform interneurons are wide with a burst index at or below it; 89.9% of dentate narrow-waveform interneurons (222 of 247) are below 0.425 ms, just short of 90%. Two of the three groups meet the criterion.

## interneuron-rates

> Narrow-waveform interneurons fire faster than wide-waveform interneurons (medians 7.97 and 2.11 Hz; Kruskal-Wallis P < 0.00001).

Readings: (a) spikes over the span of the recording, (b) the rate in `awake` periods. The paper's counts of interneurons (248 narrow, 268 wide) are dentate units, with CA3 units "separately classified" (Results, Identification..., paragraph 1); the dentate units are the main set, and all labelled interneurons are shown as a variant.

In [ ]:
rows = []
for setname, mask in [("DG", DG_I), ("DG and CA3", U.type.isin(["nI", "wI"]))]:
    for rd, col in [("a span", "rate"), ("b awake", "rate_awake")]:
        n = U[mask & (U.type == "nI")][col]
        w = U[mask & (U.type == "wI")][col]
        rows.append(dict(units=setname, reading=rd, n_nI=n.notna().sum(), n_wI=w.notna().sum(),
                         median_nI=round(n.median(), 3), median_wI=round(w.median(), 3), KW_P=kw(n, w),
                         nI_in_tol=abs(n.median() - 7.97) <= 0.797, wI_in_tol=abs(w.median() - 2.11) <= 0.211))
print(pd.DataFrame(rows).to_string())
print("\nby subject (DG, reading a):")
print(U[DG_I].groupby(["subject", "type"]).rate.median().unstack().round(2).to_string())

**Result: reproduced.** Among dentate interneurons, the narrow-waveform median rate is 7.65 Hz and the wide-waveform one 2.05 Hz over the recording span (reading a), both within tolerance of 7.97 and 2.11 Hz, with Kruskal-Wallis P = 1.4e-13. In `awake` periods (reading b) the medians are 9.23 and 2.87 Hz, outside tolerance. With CA3 interneurons included, the narrow-waveform median falls to 5.80 Hz.

## gc-narrower-waveform

> The waveform of granule cells is narrower than that of mossy cells.

In [ ]:
g = U[(U.type == "GC") & U.tp_ms.notna()]
m = U[(U.type == "MC") & U.tp_ms.notna()]
print(f"trough-to-peak median GC {g.tp_ms.median():.3f} ms (n={len(g)}), MC {m.tp_ms.median():.3f} ms (n={len(m)}); Mann-Whitney P = {mwu(g.tp_ms, m.tp_ms):.3g}")
print(f"peak at the end of the stored window: GC {g.tp_at_end.mean():.3f}, MC {m.tp_at_end.mean():.3f}")
gw = U[(U.type == "GC") & U.tp_wb_ms.notna()]; mw_ = U[(U.type == "MC") & U.tp_wb_ms.notna()]
print(f"variant, wide-band waveform at the site (17 files): GC {gw.tp_wb_ms.median():.3f} (n={len(gw)}), MC {mw_.tp_wb_ms.median():.3f} (n={len(mw_)}), P = {mwu(gw.tp_wb_ms, mw_.tp_wb_ms):.3g}")
print("\nby subject (median ms):")
print(U[U.type.isin(["GC", "MC"])].groupby(["subject", "type"]).tp_ms.median().unstack().round(3).to_string())

**Result: reproduced.** The GC median trough-to-peak latency (0.710 ms) is below the MC median (0.740 ms), Mann-Whitney P = 3.6e-14. The latency is a lower bound for many units, as the peak falls at the end of the stored 1.6 ms for 44% of GC and 69% of MC; with the wide-band waveform at the site (17 files) the medians are 0.700 and 0.710 ms and the difference is not significant (P = 0.43).

## optotag-drd2

> In DRD2-Cre::Ai35 mice, 29 of 107 putative mossy cells were suppressed by green light, against none of 202 putative granule cells, 8 of 68 wide-waveform and none of 69 narrow-waveform interneurons.

Units of DRD2-Cre::Ai35 files that hold 20 ms pulses; suppressed when the rate in +2 to +15 ms after onset is at least 20% below the baseline (Methods). Baseline (i) the 20 ms before the pulse, (ii) 50 to 200 ms after onset. Interneurons are the dentate ones, as for the paper's counts. The dataset's `drd2_tagged` column (the authors' own result) is shown beside.

In [ ]:
d = U[(U.genotype == "DRD2-Cre::Ai35") & U.light_resp.notna() & U.type.isin(["GC", "MC", "nI", "wI"])
      & ((U.region == "DG") | U.type.isin(["GC", "MC"]))].copy()
d["sup_pre"] = d.light_resp <= 0.8 * d.light_pre
d["sup_post"] = d.light_resp <= 0.8 * d.light_post
d.loc[d.light_pre == 0, "sup_pre"] = False
d.loc[d.light_post == 0, "sup_post"] = False
tab = d.groupby("type").agg(n=("key", "size"), suppressed_pre=("sup_pre", "sum"), suppressed_post=("sup_post", "sum"),
                            authors_tagged=("drd2_tagged", "sum"))
tab["frac_pre"] = (tab.suppressed_pre / tab.n).round(3)
tab["frac_post"] = (tab.suppressed_post / tab.n).round(3)
print(tab.to_string())
print("\npaper: MC 29/107 (0.271), GC 0/202, wI 8/68, nI 0/69")
print("DRD2 files with 20 ms pulses:", files[(files.genotype == "DRD2-Cre::Ai35") & (files.pulses20 > 0)].shape[0],
      "of", (files.genotype == "DRD2-Cre::Ai35").sum(), "; pulses per file: median",
      files[(files.genotype == "DRD2-Cre::Ai35") & (files.pulses20 > 0)].pulses20.median())
mcs = d[d.type == "MC"]
print("\nMC tagged by the authors: suppressed under baseline (i)", mcs[mcs.drd2_tagged].sup_pre.sum(), "of", mcs.drd2_tagged.sum(),
      "; response/baseline (i) median", round((mcs[mcs.drd2_tagged].light_resp / mcs[mcs.drd2_tagged].light_pre).median(), 3),
      "against", round((mcs[~mcs.drd2_tagged].light_resp / mcs[~mcs.drd2_tagged].light_pre).median(), 3), "for untagged MC")
print("\nby subject (MC suppressed, baseline i / ii / n):")
print(d[d.type == "MC"].groupby("subject").agg(n=("key", "size"), pre=("sup_pre", "sum"), post=("sup_post", "sum")).to_string())

**Result: partially reproduced.** Under baseline (i), the 20 ms before each pulse, 68 of 106 MC (0.642) are suppressed, against 6 of 198 GC (0.030), 15 of 62 wide-waveform and 4 of 68 narrow-waveform dentate interneurons. The GC fraction is below 0.05 and the MC fraction exceeds four times it, but the MC fraction is far above 0.271 (29 of 107); under baseline (ii) it is 0.670. The 29 MC the authors tagged are all among those suppressed, so the criterion's 20% decrease, without a significance test, admits more units than the paper's.

## mc-more-active

> Mossy cells are more active than granule cells during both waking and NREM sleep (Kruskal-Wallis P < 0.0001 for each).

Per-unit rates in each state from the `brain_states` table. Wake is read as `awake` alone and as `awake` and `quiet` together.

In [ ]:
g, m = U[U.type == "GC"], U[U.type == "MC"]
for col, name in [("rate_awake", "wake (awake)"), ("rate_wake_aq", "wake (awake+quiet)"), ("rate_nrem", "NREM")]:
    print(f"{name:20s} GC median {g[col].median():.3f} (n={g[col].notna().sum()})  MC median {m[col].median():.3f} (n={m[col].notna().sum()})  KW P = {kw(g[col], m[col]):.3g}")
print("\nby subject (median Hz, awake / NREM):")
print(U[U.type.isin(["GC", "MC"])].groupby(["subject", "type"])[["rate_awake", "rate_nrem"]].median().unstack().round(3).to_string())

**Result: reproduced.** MC fire faster than GC in wake (`awake`: median 0.726 against 0.079 Hz, Kruskal-Wallis P = 4e-102; `awake` and `quiet`: P = 3e-100) and in NREM (0.907 against 0.398 Hz, P = 1e-23).

## gc-nrem-highest and ca3-wake-highest

> Granule cells fire most during NREM sleep and at similar rates during wake and REM.

> CA3 pyramidal neurons are most active during wake and least active during REM.

Kruskal-Wallis across the three states, then pairwise Mann-Whitney tests with Bonferroni correction (three comparisons). Units with no time in a state are left out of that state.

In [ ]:
def states_test(t):
    x = U[U.type == t]
    out = []
    for wcol in ["rate_awake", "rate_wake_aq"]:
        g = {"wake": x[wcol].dropna(), "nrem": x.rate_nrem.dropna(), "rem": x.rate_rem.dropna()}
        p_kw = kw(*g.values())
        pw = {f"{a}-{b}": min(1.0, 3 * mwu(g[a], g[b])) for a, b in [("nrem", "wake"), ("nrem", "rem"), ("wake", "rem")]}
        out.append(dict(type=t, wake=wcol, **{f"median_{k}": round(v.median(), 3) for k, v in g.items()},
                        **{f"n_{k}": len(v) for k, v in g.items()}, KW_P=p_kw, **{f"Padj_{k}": v for k, v in pw.items()}))
    return out
st = pd.DataFrame(states_test("GC") + states_test("CA3"))
print(st.T.to_string())
print("\nby subject, CA3 (median Hz):")
print(U[U.type == "CA3"].groupby("subject")[["rate_awake", "rate_nrem", "rate_rem"]].median().round(3).to_string())
print("\nby subject, GC (median Hz):")
print(U[U.type == "GC"].groupby("subject")[["rate_awake", "rate_nrem", "rate_rem"]].median().round(3).to_string())

**Result: gc-nrem-highest partially reproduced; ca3-wake-highest not reproduced.** GC fire most in NREM (median 0.398 Hz, against 0.079 in wake and 0.205 in REM, every pairwise adjusted P < 1e-10 against NREM), as claimed, but their wake and REM rates differ (adjusted P near 0), under both readings of wake. For CA3 pyramidal cells REM is the lowest (0.305 Hz), but wake (0.450 Hz) is not the highest: the NREM median is 0.473 Hz, and wake and NREM do not differ (adjusted P = 0.77 with `awake`, 0.69 with `awake` and `quiet`).

## nrem-wake-ratio

> Granule cells increase their firing in NREM sleep compared with waking, while mossy cells fire at equivalent rates in the two states.

In [ ]:
for wcol in ["rate_awake", "rate_wake_aq"]:
    x = U[U.type.isin(["GC", "MC"]) & (U[wcol] > 0) & U.rate_nrem.notna()].copy()
    x["ratio"] = x.rate_nrem / x[wcol]
    g, m = x[x.type == "GC"], x[x.type == "MC"]
    print(f"wake = {wcol}: GC median ratio {g.ratio.median():.3f} (n={len(g)}), MC {m.ratio.median():.3f} (n={len(m)}), "
          f"Mann-Whitney P = {mwu(g.ratio, m.ratio):.3g}; MC NREM vs wake Wilcoxon P = {stats.wilcoxon(m.rate_nrem, m[wcol]).pvalue:.3g}")
x = U[U.type.isin(["GC", "MC"]) & (U.rate_awake > 0)].copy(); x["ratio"] = x.rate_nrem / x.rate_awake
print("\nby subject (median ratio, wake = awake):")
print(x.groupby(["subject", "type"]).ratio.median().unstack().round(2).to_string())

**Result: reproduced.** The GC median ratio of NREM to wake rate is 4.67 (above 1), against 1.09 for MC, Mann-Whitney P = 1e-130 (wake as `awake`; 4.31 and 1.10 with `quiet`). The MC NREM rate does differ from wake by a Wilcoxon test (P = 0.0002), though their median ratio is near 1.

## theta-modulation and gamma-modulation

> Granule cells are more strongly phase-modulated by theta oscillations than mossy cells and fire at later theta phases (Kruskal-Wallis P < 0.0001 for both).

> Granule cells are more strongly phase-modulated by gamma oscillations than mossy cells and fire at earlier gamma phases (Kruskal-Wallis P < 0.0001 for both).

Phases from the file's hilus reference channel (`hilus_reference`, the channel the lab used for theta and gamma phase) in `awake` periods, up to one hour per file, where the band amplitude exceeds median + 0.5 SD (theta) or + 1 SD (gamma). Units with Rayleigh P < 0.05 and kappa > 0.1. Phase readings (a) (-pi, pi] with 0 at the LFP peak, (b) [0, 2 pi).

In [ ]:
print("awake seconds used per file: median", np.nanmedian([r.get("phase_seconds", np.nan) for r in R if r["subject"] != S.EXCLUDED_SUBJECT]))
for band in ["theta", "gamma"]:
    x = U[U.type.isin(["GC", "MC"]) & U[f"{band}_mrl"].notna()]
    sig = x[(x[f"{band}_p"] < 0.05) & (x[f"{band}_kappa"] > 0.1)]
    g, m = sig[sig.type == "GC"], sig[sig.type == "MC"]
    pa = g[f"{band}_phase"], m[f"{band}_phase"]
    pb = np.mod(pa[0], 2 * np.pi), np.mod(pa[1], 2 * np.pi)
    cm = lambda p: np.angle(np.mean(np.exp(1j * p)))
    print(f"\n{band}: units with phases GC {(x.type=='GC').sum()}, MC {(x.type=='MC').sum()}; passing Rayleigh and kappa GC {len(g)}, MC {len(m)}")
    print(f"  MRL median GC {g[f'{band}_mrl'].median():.3f}, MC {m[f'{band}_mrl'].median():.3f}, KW P = {kw(g[f'{band}_mrl'], m[f'{band}_mrl']):.3g}")
    print(f"  circular mean phase (rad) GC {cm(pa[0]):.3f}, MC {cm(pa[1]):.3f}; GC minus MC = {np.angle(np.exp(1j*(cm(pa[0])-cm(pa[1])))):.3f} rad")
    print(f"  reading a, (-pi, pi]: median GC {pa[0].median():.3f}, MC {pa[1].median():.3f}, KW P = {kw(*pa):.3g}")
    print(f"  reading b, [0, 2 pi): median GC {pb[0].median():.3f}, MC {pb[1].median():.3f}, KW P = {kw(*pb):.3g}")
    print(f"  all units with phases (no Rayleigh selection): MRL median GC {x[x.type=='GC'][f'{band}_mrl'].median():.3f}, MC {x[x.type=='MC'][f'{band}_mrl'].median():.3f}")
    print("  by subject (MRL median of passing units):")
    print(sig.groupby(["subject", "type"])[f"{band}_mrl"].median().unstack().round(3).to_string())

**Result: theta-modulation reproduced; gamma-modulation reproduced.** Among units passing the Rayleigh and kappa test, the theta MRL is higher for GC (median 0.429) than MC (0.347), P = 7e-16, and GC fire later in the theta cycle (circular mean 0.536 against 0.246 rad, reading a medians 0.508 and 0.201, P = 8e-13). The gamma MRL is higher for GC (0.323 against 0.217, P = 3e-34), and GC fire earlier in the gamma cycle (circular mean -0.470 against -0.062 rad, reading a P = 2e-23). Under phase reading b ([0, 2 pi)) neither phase difference is significant (P = 0.59 and 0.67), as the phases straddle 0.

## gamma-rate-correlation

> Slow-firing neurons are more strongly coupled to gamma than fast-firing ones (GC r = -0.176, MC r = -0.184, P < 0.0001 for each).

Rate is spikes over the recording span. Readings (a) all GC or MC with a gamma MRL, (b) only the units passing Rayleigh P < 0.05 and kappa > 0.1.

In [ ]:
for rd, sel in [("a all", lambda x: x), ("b passing", lambda x: x[(x.gamma_p < 0.05) & (x.gamma_kappa > 0.1)])]:
    for t in ["GC", "MC"]:
        x = sel(U[(U.type == t) & U.gamma_mrl.notna()])
        r, p = stats.pearsonr(x.rate, x.gamma_mrl)
        print(f"reading {rd:10s} {t}: n={len(x)}, Pearson r = {r:.3f}, P = {p:.3g}")
x = U[(U.type.isin(["GC", "MC"])) & U.gamma_mrl.notna()]
print("\nvariant, Spearman (reading a): GC rho = %.3f, MC rho = %.3f" % (stats.spearmanr(x[x.type=='GC'].rate, x[x.type=='GC'].gamma_mrl)[0],
                                                                         stats.spearmanr(x[x.type=='MC'].rate, x[x.type=='MC'].gamma_mrl)[0]))
print("gamma spike counts used per unit: median GC", x[x.type=='GC'].gamma_n.median(), ", MC", x[x.type=='MC'].gamma_n.median())

**Result: not reproduced.** Both correlations are negative, but neither reading puts both r within tolerance or gives MC P < 0.0001: reading (a) GC r = -0.234 (P = 2e-8), MC r = -0.119 (P = 0.008); reading (b) GC r = -0.413 (P = 7e-15), MC r = -0.182 (P = 0.0002).

## ds2-recruitment

> Granule cells are more effectively recruited by type 2 dentate spikes than mossy cells and fire earlier (Kruskal-Wallis P < 0.0001 for both).

Histograms of spikes around the DS2 peaks stored in each file (2 ms bins, +-100 ms). Peak ratio: largest rate within +-20 ms over the mean rate at 50 to 100 ms on both sides; peak time: centre of that bin.

In [ ]:
x = U[U.type.isin(["GC", "MC"]) & U.ds2_ratio.notna() & np.isfinite(U.ds2_ratio)]
g, m = x[x.type == "GC"], x[x.type == "MC"]
print(f"DS2 per file: median {files[files.subject != S.EXCLUDED_SUBJECT].n_ds2.median():.0f}, files with none: {(files[files.subject != S.EXCLUDED_SUBJECT].n_ds2 == 0).sum()}")
print(f"peak ratio median GC {g.ds2_ratio.median():.2f} (n={len(g)}), MC {m.ds2_ratio.median():.2f} (n={len(m)}); KW P = {kw(g.ds2_ratio, m.ds2_ratio):.3g}")
print(f"peak time median GC {g.ds2_peak_ms.median():.1f} ms, MC {m.ds2_peak_ms.median():.1f} ms; KW P = {kw(g.ds2_peak_ms, m.ds2_peak_ms):.3g}")
print(f"ratio with DS2 times shifted by 0.5 s: GC {g.ds2_ratio_shift.median():.2f}, MC {m.ds2_ratio_shift.median():.2f}")
print("dropped (no DS2 or no baseline spikes):", U[U.type.isin(["GC", "MC"])].shape[0] - len(x))
print("\nby subject (medians):")
print(x.groupby(["subject", "type"])[["ds2_ratio", "ds2_peak_ms"]].median().unstack().round(2).to_string())

**Result: reproduced.** The GC median peak ratio around DS2 (6.70) exceeds the MC one (5.04), Kruskal-Wallis P = 1.9e-6, and GC peak earlier (median -3 ms against +1 ms), P = 3e-8. With the DS2 times shifted by 0.5 s the ratios fall to 1.69 and 1.63.

## mc-deeper

> Mossy cells are located deeper than granule cells relative to the reversal of the type 2 dentate spike (DS2) (Mann-Whitney P < 0.001).

Depth is read from the DS2 amplitude at each unit's site (positive below the reversal, in the hilus). Main: the 17 files with the wide-band signal, the site being the channel where the unit's spike is largest in the wide band. Variant: the channel of the largest stored waveform through the electrodes table, on every file with waveforms. The DS2 amplitude is the LFP at the DS2 peak minus its mean at -36 to -16 ms (the baseline of the Methods' DS2 detection), averaged over up to 600 DS2 events of the file.

In [ ]:
print("DS2 events averaged per file: median", np.median([r.get("ds2_profile_n", 0) for r in R if r.get("ds2_profile") is not None]))
for name, col in [("wide-band site (main)", "ds2_amp_wb"), ("stored-waveform site (variant)", "ds2_amp_table")]:
    x = U[U.type.isin(["GC", "MC"]) & U[col].notna()]
    g, m = x[x.type == "GC"][col], x[x.type == "MC"][col]
    print(f"{name}: GC median {g.median():.1f} uV (n={len(g)}), MC median {m.median():.1f} uV (n={len(m)}), Mann-Whitney P = {mwu(g, m):.3g}")
    print(f"   positive DS2 at the site: GC {(g > 0).mean():.2f}, MC {(m > 0).mean():.2f}")
x = U[U.type.isin(["GC", "MC"]) & U.ds2_amp_wb.notna()]
print("\nby file (wide-band site, median uV):")
print(x.groupby(["path", "type"]).ds2_amp_wb.median().unstack().round(0).rename(index=lambda p: p.split("ses-")[1][:20]).to_string())

**Result: reproduced.** At the wide-band site, the MC median DS2 amplitude (1144 uV, n = 240) exceeds the GC median (451 uV, n = 136), Mann-Whitney P = 5e-44; every MC and 82% of GC sites have a positive DS2. The variant with the stored-waveform site on all files gives 1130 against 401 uV, P = 9e-114.

## gc-mc-classification

> K-means clustering on the DS2 amplitude at the unit's site, the NREM to wake rate ratio and two principal components of the waveform's second derivative separates excitatory dentate units into granule cells and mossy cells.

On the GC and MC of the 17 wide-band files. Features as the criterion gives them: DS2 amplitude at the wide-band site; the NREM to wake (`awake`) rate ratio; w-PC1 and w-PC2 of the second derivative (trough to 0.8 ms after it) of the wide-band waveform at the site, normalised to its trough and upsampled to 100 kHz. Standardised, then k-means with two clusters, 50 starts, seed from the claim; the seed plus 1000 to 4000 are shown too.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

x = U[U.type.isin(["GC", "MC"]) & U.ds2_amp_wb.notna() & (U.rate_awake > 0) & U.rate_nrem.notna()].copy()
x = x[x.d2_wb.apply(lambda v: isinstance(v, np.ndarray) and np.isfinite(v).all())]
D2 = np.vstack(x.d2_wb.values)
pcs = PCA(2).fit_transform(D2)
feat = np.column_stack([x.ds2_amp_wb.values, (x.rate_nrem / x.rate_awake).values, pcs])
Z = (feat - feat.mean(0)) / feat.std(0)
lab = (x.type == "MC").values
print("units:", len(x), "GC", (~lab).sum(), "MC", lab.sum())
res_seed = {}
for off in [0, 1000, 2000, 3000, 4000]:
    km = KMeans(2, n_init=50, random_state=(seed("gc-mc-classification") + off) % 2**32).fit(Z)
    agree = max((km.labels_ == lab).mean(), (km.labels_ != lab).mean())
    res_seed[off] = agree
    print(f"seed + {off}: agreement {agree:.3f}")
for name, cols in [("without waveform PCs", [0, 1]), ("DS2 only", [0]), ("ratio only", [1]), ("waveform PCs only", [2, 3])]:
    km = KMeans(2, n_init=50, random_state=seed("gc-mc-classification")).fit(Z[:, cols])
    print(f"variant {name}: agreement {max((km.labels_ == lab).mean(), (km.labels_ != lab).mean()):.3f}")
xs = U[U.type.isin(["GC", "MC"]) & U.ds2_amp_table.notna() & (U.rate_awake > 0) & U.rate_nrem.notna()].copy()
# the stored 1.6 ms often ends before 0.8 ms after the trough; the first 0.4 ms after it is used here
xs = xs[xs.d2.apply(lambda v: isinstance(v, np.ndarray) and np.isfinite(v[:41]).all())]
F2 = np.column_stack([xs.ds2_amp_table.values, (xs.rate_nrem / xs.rate_awake).values,
                      PCA(2).fit_transform(np.vstack([v[:41] for v in xs.d2.values]))])
Z2 = (F2 - F2.mean(0)) / F2.std(0)
km = KMeans(2, n_init=50, random_state=seed("gc-mc-classification")).fit(Z2)
l2 = (xs.type == "MC").values
print(f"variant, stored waveforms and table site, all files: n={len(xs)}, agreement {max((km.labels_ == l2).mean(), (km.labels_ != l2).mean()):.3f}")

**Result: reproduced.** K-means on the four standardised features agrees with the labels for 94.9% of the 376 units (136 GC, 240 MC), the same under all five seeds. The variant on all files with the stored waveforms and the table site gives 92.7%; without the waveform components 89.9%.

## Pairs: cross-correlogram tests

Every GC-MC pair recorded in the same file, and every same-type pair of GC, MC and CA3 pyramidal cells, was tested with the automatic test of the Methods (0.5 ms bins, baseline from a 10 ms SD Gaussian, 99.9999th Poisson percentile, two consecutive bins at +1.5 to +5 ms or -1 to +1 ms). The paper's manual inspection is not repeated. The authors' own tables of monosynaptic and zero-lag pairs, which the files hold, are shown beside as a reference.

In [ ]:
P = []
for r in R:
    if r["subject"] == S.EXCLUDED_SUBJECT:
        continue
    typ = {u["unit"]: u["type"] for u in r["units"]}
    for p in r["pairs"]:
        P.append(dict(path=r["path"], subject=r["subject"], **p))
P = pd.DataFrame(P)
gm = P[P.kind == "GC-MC"].copy()
print(P.groupby("kind").size().to_string())
# the authors' monosynaptic table, typed by the units' labels
A = []
for r in R:
    if r["subject"] == S.EXCLUDED_SUBJECT or r["authors_mono"] is None:
        continue
    typ = {u["unit"]: u["type"] for u in r["units"]}
    for a, b, c in r["authors_mono"]:
        A.append(dict(path=r["path"], pre=a, post=b, connection=c, pre_type=typ.get(a), post_type=typ.get(b)))
A = pd.DataFrame(A)

## monosyn-gc-to-mc

> 5.5% of granule cells monosynaptically excite mossy cells (32 pairs), whereas no mossy cell significantly excites a granule cell.

Fraction read three ways: (a) significant GC to MC pairs over all GC, (b) GC with at least one MC target over all GC, (c) significant pairs over tested pairs.

In [ ]:
n_gc = (U.type == "GC").sum()
n_gc_tested = gm.pre.groupby(gm.path).nunique().sum()
sig = gm[gm.gc_to_mc]
rev = gm[gm.mc_to_gc]
n_gc_target = sig.groupby("path").pre.nunique().sum()
print(f"GC-MC pairs tested: {len(gm)}; GC: {n_gc} (in files with an MC: {n_gc_tested})")
print(f"GC to MC pairs: {len(sig)};  MC to GC pairs: {len(rev)}")
print(f"(a) pairs / all GC = {len(sig)}/{n_gc} = {len(sig)/n_gc:.4f}")
print(f"(b) GC with an MC target / all GC = {n_gc_target}/{n_gc} = {n_gc_target/n_gc:.4f}")
print(f"(c) pairs / tested pairs = {len(sig)}/{len(gm)} = {len(sig)/len(gm):.4f}")
print(f"variant, GC in files with an MC as denominator: (a) {len(sig)/n_gc_tested:.4f}, (b) {n_gc_target/n_gc_tested:.4f}")
print(f"same shank among GC to MC pairs: {sig.same_shank.sum()} of {len(sig)}; among MC to GC: {rev.same_shank.sum()} of {len(rev)}")
ae = A[(A.connection == "excitatory")]
a_gm = ae[(ae.pre_type == "GC") & (ae.post_type == "MC")]
a_mg = ae[(ae.pre_type == "MC") & (ae.post_type == "GC")]
print(f"\nauthors' table (files that hold it): GC to MC excitatory {len(a_gm)}, MC to GC excitatory {len(a_mg)}, GC with an MC target {a_gm.groupby('path').pre.nunique().sum()}")
both = sig.merge(a_gm, on=["path", "pre", "post"])
va = sig.merge(a_gm[["path", "pre", "post"]], on=["path", "pre", "post"])  # the authors' pairs, as found here
print(f"pairs found by both this test and the authors' table: {len(both)}")
print("\nby subject (GC to MC / MC to GC / tested):")
print(gm.groupby("subject").agg(tested=("pre", "size"), gc_to_mc=("gc_to_mc", "sum"), mc_to_gc=("mc_to_gc", "sum")).to_string())

**Result: partially reproduced.** The automatic test finds 63 GC to MC pairs and 8 MC to GC pairs among 4425 tested. The fraction is 0.108 (pairs over all GC), 0.081 (GC with a target) or 0.014 (pairs over tested pairs), none within 0.055 +- 0.0055, and MC to GC pairs exist; GC to MC pairs outnumber them 63 to 8, more than four to one. The authors' table in the files holds exactly 32 GC to MC pairs and no MC to GC pair, as the paper reports, and the automatic test finds all 32; the paper's manual inspection, not repeated here, is what removes the rest.

## burst-facilitation

> Spike transmission probability from granule cells to mossy cells increases from the first to the second spike of a granule cell burst (P < 0.001).

For the detected GC to MC pairs: transmission probability (CCG counts minus the Gaussian baseline at +1.5 to +5 ms, per GC spike) for the first and second spikes of bursts (spikes at most 6 ms apart).

In [ ]:
b = sig[sig.n_burst > 0]
print(f"pairs: {len(b)} (bursts per GC: median {b.n_burst.median():.0f})")
print(f"transmission probability median: first {b.tp_first.median():.4f}, second {b.tp_second.median():.4f}; all spikes {sig.tp_all.median():.4f}")
print(f"second > first in {(b.tp_second > b.tp_first).sum()} of {len(b)}; paired Wilcoxon P = {stats.wilcoxon(b.tp_second, b.tp_first).pvalue:.3g}")
print(f"variant, the authors' {len(va)} pairs: first {va.tp_first.median():.4f}, second {va.tp_second.median():.4f}, second > first in {(va.tp_second > va.tp_first).sum()}, Wilcoxon P = {stats.wilcoxon(va.tp_second, va.tp_first).pvalue:.3g}")
print("\nby subject (median second minus first):")
print(b.assign(d=b.tp_second - b.tp_first).groupby("subject").agg(n=("d", "size"), median_diff=("d", "median")).round(4).to_string())

**Result: reproduced.** Across the 63 detected pairs, the transmission probability of the second burst spike (median 0.032) exceeds that of the first (0.0067) in 56 pairs, paired Wilcoxon P = 2.9e-9. On the authors' 32 pairs, 32 of 32, P = 4.7e-10 (printed above).

## state-transmission

> GC to MC spike transmission is lower during NREM sleep than during waking (n = 31, P < 0.005).

Transmission probability as in burst-facilitation, from GC spikes in `awake` periods against GC spikes in `nrem` periods, per detected pair.

In [ ]:
s2 = sig[(sig.n_wake > 0) & (sig.n_nrem > 0)]
print(f"pairs: {len(s2)}; GC spikes per pair in wake median {s2.n_wake.median():.0f}, in NREM {s2.n_nrem.median():.0f}")
print(f"median transmission wake {s2.tp_wake.median():.4f}, NREM {s2.tp_nrem.median():.4f}; NREM lower in {(s2.tp_nrem < s2.tp_wake).sum()} of {len(s2)}")
print(f"paired Wilcoxon P = {stats.wilcoxon(s2.tp_nrem, s2.tp_wake).pvalue:.3g}; one-sided (NREM lower) P = {stats.wilcoxon(s2.tp_nrem, s2.tp_wake, alternative='less').pvalue:.3g}")
print(f"variant, the authors' {len(va)} pairs: wake {va.tp_wake.median():.4f}, NREM {va.tp_nrem.median():.4f}, NREM lower in {(va.tp_nrem < va.tp_wake).sum()}, Wilcoxon P = {stats.wilcoxon(va.tp_nrem, va.tp_wake).pvalue:.3g}")
print("\nby subject:")
print(s2.assign(d=s2.tp_nrem - s2.tp_wake).groupby("subject").agg(n=("d", "size"), median_nrem_minus_wake=("d", "median")).round(4).to_string())

**Result: not reproduced.** Across the 63 detected pairs, the median transmission is 0.0125 in wake and 0.0111 in NREM, lower in NREM for 32 of 63 pairs, paired Wilcoxon P = 0.52. On the authors' 32 pairs (printed above) NREM is lower in 19, P = 0.018, which would meet only the partial level.

## zero-lag-synchrony

> Synchronous co-activation within 1 ms is found in 23% of mossy cells and 9% of CA3 pyramidal cells, but in fewer than 0.2% of granule cell pairs.

The Methods' test at -1 to +1 ms on all same-type pairs recorded together. MC and CA3 read as (a) fractions of pairs, (b) fractions of cells with at least one synchronous partner; GC as a fraction of pairs (the claim holds for any value up to 0.0022). The files now say which shank each unit is on, so pairs on different shanks are shown as a variant.

In [ ]:
rows = []
for kind, t in [("GC-GC", "GC"), ("MC-MC", "MC"), ("CA3-CA3", "CA3")]:
    x = P[P.kind == kind]
    for setname, xx in [("all pairs", x), ("different shanks", x[~x.same_shank]), ("same shank", x[x.same_shank])]:
        cells = pd.concat([xx[xx.sync][["path", "pre"]].rename(columns={"pre": "u"}), xx[xx.sync][["path", "post"]].rename(columns={"post": "u"})]).drop_duplicates()
        n_cells = (U.type == t).sum()
        rows.append(dict(kind=kind, pairs=setname, n_pairs=len(xx), sync_pairs=int(xx.sync.sum()),
                         frac_pairs=round(xx.sync.mean(), 4) if len(xx) else np.nan,
                         cells_with_partner=len(cells), frac_cells=round(len(cells) / n_cells, 5)))
print(pd.DataFrame(rows).to_string())
# authors' zero-lag table
Z = []
for r in R:
    if r["subject"] == S.EXCLUDED_SUBJECT or r["authors_sync"] is None:
        continue
    typ = {u["unit"]: u["type"] for u in r["units"]}
    for a, bb in r["authors_sync"]:
        Z.append(dict(path=r["path"], a=a, b=bb, ta=typ.get(a), tb=typ.get(bb)))
Z = pd.DataFrame(Z)
print("\nauthors' zero-lag table (files that hold it): pairs by type")
print(pd.crosstab(Z.ta, Z.tb).to_string())
for t in ["GC", "MC", "CA3"]:
    zz = Z[(Z.ta == t) & (Z.tb == t)]
    cells = set(zip(zz.path, zz.a)) | set(zip(zz.path, zz.b))
    print(f"{t}: same-type pairs {len(zz)}, cells with a partner {len(cells)} of {(U.type == t).sum()} ({len(cells)/(U.type == t).sum():.3f})")

**Result: partially reproduced.** GC pairs are synchronous in 0.0013 of pairs (4 of 3105), within the claim's bound of 0.0022. As fractions of pairs, MC 0.082 and CA3 0.011; as fractions of cells with a synchronous partner, MC 0.429 and CA3 0.081; none is within tolerance of 0.23 and 0.09 (CA3 0.0810 lies just outside 0.081). Under reading (b) both exceed the GC pair fraction by more than ten times. No same-shank pair is synchronous by this test. The authors' zero-lag table gives 111 of 503 MC (0.221) and 30 of 568 CA3 (0.053) with a same-type partner, and no GC pair.

## T-maze: maps and fields

Runs on the circular T-maze, from entry into the central arm to within 10 cm of the reward port (18 cm from the end of the 102 cm side arm), light-free trials only (claims file header). The lab's `arm_start` lies about 27 cm into the 65 cm central arm (printed below), so each run starts at the mouse's last entry into the central arm before it (an assumption: the Methods start the analysis at the gate opening, which the files do not mark). Linearised: central arm 0 to 65 cm, then the left or the right side arm, 1.86 cm bins, speed > 3 cm/s, 5.57 cm Gaussian (Methods). Fields: at least 5 bins above 50% of the local maximum with a peak above 1 Hz (Methods), local maxima taken in turn from the highest.

In [ ]:
T = []
occs = {}
for r in R:
    tm = r.get("tmaze")
    if not tm or "maps" not in tm:
        continue
    occs[r["path"]] = tm["occ"]
    typ = {u["unit"]: u["type"] for u in r["units"]}
    for k, m in tm["maps"].items():
        fl = tm["fields"][k]
        T.append(dict(path=r["path"], subject=r["subject"], unit=k, type=typ[k], map=m, n_fields=len(fl), fields=fl,
                      peak=max([f[2] for f in fl]) if fl else np.nan, info=tm["info"][k],
                      split_fields=tm["splitter"][k]["n_fields"], split_p=tm["splitter"][k]["p"]))
T = pd.DataFrame(T)
runs = pd.DataFrame([dict(path=r["path"], runs=r["tmaze"]["n_runs"], left=r["tmaze"]["n_left"]) for r in R if r.get("tmaze")])
print("T-maze files:", len(runs), " with at least 4 light-free runs:", len(occs), "; runs per file median", runs.runs.median(), "range", runs.runs.min(), "-", runs.runs.max())
ast = np.concatenate([r["tmaze"].get("arm_start_cm", []) for r in R if r.get("tmaze")] or [np.zeros(0)])
if len(ast):
    print(f"position at the lab's arm_start, cm into the 65 cm central arm: median {np.median(ast):.1f}, 10th to 90th percentile {np.percentile(ast, 10):.1f} to {np.percentile(ast, 90):.1f} ({len(ast)} runs)")
print(T.groupby("type").size().to_string())
print("by subject:"); print(pd.crosstab(T.subject, T.type).to_string())

In [ ]:
# Clock check for positions: spatial information of GC, MC and CA3 in two T-maze files with spikes as
# recorded and shifted by 60 s.
for path in T.groupby("path").size().sort_values().index[-2:]:
    a = [x for x in assets if x["path"] == path][0]
    f = S.open_asset(a["asset_id"])
    u = f["units"]
    sti = u["spike_times_index"][:].astype(np.int64); s0 = np.concatenate([[0], sti[:-1]])
    keys = T[T.path == path].unit.tolist()
    spk = {k: u["spike_times"][s0[k]:sti[k]] for k in keys}
    runs_ = S.tmaze_runs(f)
    info = {}
    for shift in [0, 60]:
        occ, sc, _ = S.tmaze_maps(runs_, {k: v + shift for k, v in spk.items()})
        ok = occ > 0
        info[shift] = [S.skaggs_info(occ, S.tm_smooth(np.where(ok, sc[k] / np.where(ok, occ, 1), np.nan), occ)) for k in keys]
    f.close()
    print(path.split("ses-")[1][:20], f"units {len(keys)}: median information {np.nanmedian(info[0]):.3f} bits/s as recorded, {np.nanmedian(info[60]):.3f} with spikes shifted 60 s; higher as recorded for {np.sum(np.array(info[0]) > np.array(info[60]))} units")

## tmaze-central-active

> On the circular T-maze, the fraction of active neurons (above 1 Hz) is higher in the central arm than in the side arms, for granule cells, mossy cells and CA3 pyramidal cells (Kruskal-Wallis P < 0.00001 for each).

Fraction of each type's units above 1 Hz in each bin of the linearised maze (units with a value in the bin); central-arm bins against side-arm bins.

In [ ]:
for t in ["GC", "MC", "CA3"]:
    M = np.vstack(T[T.type == t]["map"].values)
    frac = np.nansum(M > 1, axis=0) / np.sum(np.isfinite(M), axis=0)
    cen, side = frac[:S.N_CEN], frac[S.N_CEN:]
    cen, side = cen[np.isfinite(cen)], side[np.isfinite(side)]
    print(f"{t}: units {len(M)}; median fraction active central {np.median(cen):.3f} ({len(cen)} bins), side {np.median(side):.3f} ({len(side)} bins); KW P = {kw(cen, side):.3g}")
    for sub, g in T[T.type == t].groupby("subject"):
        Mg = np.vstack(g["map"].values)
        with np.errstate(invalid="ignore"):
            fg = np.nansum(Mg > 1, axis=0) / np.sum(np.isfinite(Mg), axis=0)
        print(f"     {sub}: n={len(Mg)}, central {np.nanmedian(fg[:S.N_CEN]):.3f}, side {np.nanmedian(fg[S.N_CEN:]):.3f}")

**Result: partially reproduced.** The fraction of active units is higher in the central arm only for MC (median 0.328 against 0.239, Kruskal-Wallis P = 5e-10). For GC (0.018 against 0.022, P = 0.41) and CA3 (0.233 against 0.219, P = 0.21) it is not.

## tmaze-multiple-fields

> Few granule cells have two or more place fields on the T-maze (6.6%), against 57.8% of mossy cells and 36.1% of CA3 pyramidal cells (Kolmogorov-Smirnov P < 0.0001).

Fractions (a) of all units, (b) of units with at least one field.

In [ ]:
for t in ["GC", "MC", "CA3"]:
    x = T[T.type == t]
    print(f"{t}: n={len(x)}, with a field {(x.n_fields > 0).sum()}, two or more {(x.n_fields >= 2).sum()}; "
          f"(a) {(x.n_fields >= 2).mean():.3f}, (b) {(x.n_fields >= 2).sum() / max((x.n_fields > 0).sum(), 1):.3f}; field counts {x.n_fields.value_counts().sort_index().to_dict()}")
print("KS GC vs MC on field counts: P = %.3g" % stats.ks_2samp(T[T.type == "GC"].n_fields, T[T.type == "MC"].n_fields).pvalue)
print("paper: GC 0.066, MC 0.578, CA3 0.361")
print("\nby subject, fraction with two or more (a):")
print(T.groupby(["subject", "type"]).n_fields.apply(lambda v: (v >= 2).mean()).unstack().round(2).to_string())

**Result: partially reproduced.** Of all units, 4.9% of GC, 62.6% of MC and 48.6% of CA3 have two or more fields (reading a; of units with a field, 27.5%, 81.9% and 74.1%). Only the MC fraction is within tolerance (GC 0.049 against 0.066 +- 0.0066, CA3 0.486 against 0.361 +- 0.036); GC have the lowest fraction and the GC against MC Kolmogorov-Smirnov test gives P = 2e-37.

## tmaze-peak-info

> Granule cells have the lowest within-field peak firing rates (GC against MC, Kolmogorov-Smirnov P < 0.003) and the lowest spatial information of the three types (P < 0.0005).

Within-field peak rate: the highest field peak of each unit with a field. Spatial information (bits/s, Skaggs) of every unit, from the smoothed linearised map.

In [ ]:
pk = {t: T[(T.type == t)].peak.dropna() for t in ["GC", "MC", "CA3"]}
inf = {t: T[(T.type == t)]["info"].dropna() for t in ["GC", "MC", "CA3"]}
for t in pk:
    print(f"{t}: peak median {pk[t].median():.2f} Hz (n={len(pk[t])}), information median {inf[t].median():.3f} bits/s (n={len(inf[t])})")
print("KS peak GC vs MC P = %.3g; GC vs CA3 P = %.3g" % (stats.ks_2samp(pk["GC"], pk["MC"]).pvalue, stats.ks_2samp(pk["GC"], pk["CA3"]).pvalue))
print("KS information GC vs MC P = %.3g; GC vs CA3 P = %.3g" % (stats.ks_2samp(inf["GC"], inf["MC"]).pvalue, stats.ks_2samp(inf["GC"], inf["CA3"]).pvalue))

**Result: reproduced.** GC have the lowest within-field peak rate (median 2.54 Hz, against 4.89 for MC; KS P = 0.0005) and the lowest spatial information (0.033 bits/s, against 0.424 for MC and 0.240 for CA3; KS P = 2e-41 and 2e-28).

## tmaze-splitters

> The fraction of splitter cells, whose central-arm firing predicts the coming turn, is comparable in granule cells, mossy cells and CA3 pyramidal cells.

A splitter has a central-arm field (from the left-run or the right-run map) whose per-run rates in the field's bins differ between left and right runs by a two-way ANOVA (position x choice, choice P < 0.001; Methods). Chi-square test of splitter counts across the three types (criterion). The denominator is not given: the main reading is the units with a central-arm field (the units the Methods test); all units are shown as the other reading.

In [ ]:
T["splitter"] = T.split_p.notna() & (T.split_p < 0.001)
rows = []
for name, base in [("units with a central-arm field", T[T.split_fields > 0]), ("all units", T)]:
    ct = pd.crosstab(base.type, base.splitter)
    ct = ct.reindex(columns=[False, True], fill_value=0)
    chi = stats.chi2_contingency(ct.values)
    for t in ct.index:
        rows.append(dict(reading=name, type=t, n=int(ct.loc[t].sum()), splitters=int(ct.loc[t, True]), fraction=round(ct.loc[t, True] / ct.loc[t].sum(), 3), chi2_P=chi.pvalue))
print(pd.DataFrame(rows).to_string())

**Result: reproduced.** Splitters are 0 of 17 GC, 5 of 158 MC and 3 of 127 CA3 units with a central-arm field, chi-square P = 0.72; over all units P = 0.098. Both readings find no difference at P < 0.05. Splitters are few under the Methods' P < 0.001 (8 in all).

## Open fields: maps and place cells

Open fields A ('Old'), B ('New') and A' ('Oldlast'), with the curtain and other-room sessions used only for inheritance-openfield. Positions in cm (46 cm arena), 0.92 cm pixels, speed > 3 cm/s, 1.84 cm Gaussian (Methods). Place cells: in at least one of A, B, A', a first-half against second-half map correlation of at least 0.5 and a field of at least 100 pixels above the larger of 20% of the peak and 1 Hz (criterion).

In [ ]:
O = []
for r in R:
    of = r.get("openfield")
    if not of or not all(k in of for k in ["Old", "New", "Oldlast"]):
        continue
    typ = {u["unit"]: u["type"] for u in r["units"]}
    for k in of["Old"]["maps"]:
        row = dict(path=r["path"], subject=r["subject"], unit=k, type=typ[k])
        pc = False
        for s in ["Old", "New", "Oldlast"]:
            hr = of[s]["half_r"][k]
            row[f"half_{s}"] = hr
            row[f"field_{s}"] = of[s]["field"][k] is not None
            pc |= bool(np.isfinite(hr) and hr >= 0.5 and of[s]["field"][k] is not None)
        row["place_cell"] = pc
        mA, mB, mA2 = (of[s]["maps"][k] for s in ["Old", "New", "Oldlast"])
        row["rAA"], row["rAB"], row["rBA"] = S.corr_maps(mA, mA2), S.corr_maps(mA, mB), S.corr_maps(mB, mA2)
        row["remap"] = row["rAA"] - np.mean([row["rAB"], row["rBA"]])
        O.append(row)
O = pd.DataFrame(O)
print("files with A, B and A':", O.path.nunique(), "subjects:", O.subject.nunique())
print(O.groupby("type").agg(units=("unit", "size"), place_cells=("place_cell", "sum")).to_string())
print("paper: 247 GC, 181 MC, 317 CA3 in 5 mice; place cells 30, 111, 144")
print("by subject:"); print(O.groupby(["subject", "type"]).place_cell.agg(["size", "sum"]).unstack().to_string())

## remap-positive

> For neurons with stable place fields, the remapping index between open fields A and B is above zero in each group (Wilcoxon signed-rank; GC P < 0.05, MC and CA3 P < 0.00001).

In [ ]:
pcs = O[O.place_cell & O.remap.notna()]
for t, lvl in [("GC", 0.05), ("MC", 1e-5), ("CA3", 1e-5)]:
    x = pcs[pcs.type == t].remap
    print(f"{t}: n={len(x)}, median index {x.median():.3f}, Wilcoxon P = {stats.wilcoxon(x).pvalue:.3g} (paper's level {lvl})")

**Result: reproduced.** The remapping index of place cells is above zero in each group: GC median 0.161 (n = 24, Wilcoxon P = 0.0025), MC 0.281 (n = 102, P = 3e-10), CA3 0.233 (n = 141, P = 1e-14), each at the paper's level.

## remap-gc-lt-mc

> Granule cells with stable place fields remap less than mossy cells (Kruskal-Wallis P = 0.034; post hoc GC against MC P = 0.028).

In [ ]:
g, m, c = (pcs[pcs.type == t].remap for t in ["GC", "MC", "CA3"])
print(f"medians GC {g.median():.3f}, MC {m.median():.3f}, CA3 {c.median():.3f}; KW P = {kw(g, m, c):.3g}; "
      f"Mann-Whitney GC vs MC, Bonferroni x3: P = {min(1, 3 * mwu(g, m)):.3g}")

**Result: not reproduced.** The GC median index (0.161) is below the MC median (0.281), but neither test holds: Kruskal-Wallis across the three types P = 0.69, Bonferroni-corrected Mann-Whitney GC against MC P = 1.

## remap-fractions

> Only 36.7% of granule cells with place fields (11 of 30) remap, against 61.3% of mossy cells (68 of 111) and 55.6% of CA3 pyramidal cells (80 of 144).

In [ ]:
for t in ["GC", "MC", "CA3"]:
    x = pcs[pcs.type == t].remap
    print(f"{t}: remapping (index >= 0.2) {(x >= 0.2).sum()} of {len(x)} = {(x >= 0.2).mean():.3f}; all units of the type in A/B/A' files: {(O.type == t).sum()}")

**Result: partially reproduced.** 11 of 24 GC place cells remap (0.458), against 56 of 102 MC (0.549) and 77 of 141 CA3 (0.546). Only the CA3 fraction is within tolerance; GC have the lowest fraction. The place-cell counts are 24, 102 and 141 (paper 30, 111, 144), from 246 GC, 182 MC and 317 CA3 units in the A, B, A' files (paper 247, 181, 317).

## pv-remap

> Population-vector remapping is smaller for granule cells than for mossy cells and CA3 pyramidal cells, for populations of 5, 10, 15 and 20 place cells.

Place cells pooled across sessions per type; for each population size, 1000 random draws; population-vector correlation as the mean over pixels visited in both maps (by every cell drawn) of the Pearson correlation of the stacked rates; index PV(AA') - mean(PV(AB), PV(BA')).

In [ ]:
maps = {}
for r in R:
    of = r.get("openfield")
    if not of or not all(k in of for k in ["Old", "New", "Oldlast"]):
        continue
    for k in of["Old"]["maps"]:
        maps[(r["path"], k)] = [of[s]["maps"][k].reshape(-1) for s in ["Old", "New", "Oldlast"]]


def pv_corr(X, Y):
    ok = np.isfinite(X).all(0) & np.isfinite(Y).all(0)
    X, Y = X[:, ok], Y[:, ok]
    Xc, Yc = X - X.mean(0), Y - Y.mean(0)
    den = np.sqrt((Xc ** 2).sum(0) * (Yc ** 2).sum(0))
    with np.errstate(invalid="ignore", divide="ignore"):
        rr = (Xc * Yc).sum(0) / den
    return np.nanmean(rr) if np.isfinite(rr).any() else np.nan


def pv_draws(rng, n, ndraw=1000):
    out = {}
    for t in ["GC", "MC", "CA3"]:
        keys = [(p, u) for p, u in pcs[pcs.type == t][["path", "unit"]].itertuples(index=False)]
        vals = []
        for _ in range(ndraw):
            pick = rng.choice(len(keys), n, replace=False)
            A_, B_, A2 = (np.vstack([maps[keys[i]][j] for i in pick]) for j in range(3))
            vals.append(pv_corr(A_, A2) - np.mean([pv_corr(A_, B_), pv_corr(B_, A2)]))
        out[t] = np.array(vals)
    return out

rows = []
for off in [0, 1000, 2000, 3000, 4000]:
    rng = np.random.default_rng(seed("pv-remap") + off)
    for n in [5, 10, 15, 20]:
        d = pv_draws(rng, n)
        rows.append(dict(seed_offset=off, n=n, **{f"median_{t}": round(np.nanmedian(v), 4) for t, v in d.items()},
                         KW_P=kw(*d.values()), P_GC_MC_bonf=min(1, 3 * mwu(d["GC"], d["MC"])), P_GC_CA3_bonf=min(1, 3 * mwu(d["GC"], d["CA3"]))))
pv = pd.DataFrame(rows)
print(pv.to_string())

**Result: reproduced, but seed sensitive.** Under the claim's seed, the GC median index is below the MC and CA3 medians at every size (n = 5: 0.203 against 0.254 and 0.238; n = 20: 0.2877 against 0.2902 and 0.3461), with Kruskal-Wallis P below 1e-10. At n = 20 the GC and MC medians are within 0.003 of each other, the Bonferroni GC against MC test gives P = 0.84, and under seeds +2000 and +4000 the GC median is above the MC one (0.2874 against 0.2872; 0.2903 against 0.2888).

## inheritance-tmaze

> Mossy cells rarely inherit place fields from single presynaptic granule cells; of 10 place fields of connected granule cells, only 2 overlapped a field of the postsynaptic mossy cell.

Detected GC to MC pairs in files with T-maze runs; a GC field overlaps when it shares a bin with a field of the partner MC (bins are arm-specific, so the same arm). "Rarely" is at most a third of the GC fields (criterion).

In [ ]:
Tk = T.set_index(["path", "unit"])
rows = []
for name, pairs_ in [("automatic test", sig), ("authors' table", va)]:
    n_f = n_ov = n_pairs = 0
    gcs = set()
    for p in pairs_.itertuples():
        if (p.path, p.pre) not in Tk.index or (p.path, p.post) not in Tk.index:
            continue
        n_pairs += 1
        gf, mf = Tk.loc[(p.path, p.pre)].fields, Tk.loc[(p.path, p.post)].fields
        mbins = set(b for a, z, _ in mf for b in range(a, z + 1))
        for a, z, _ in gf:
            n_f += 1
            n_ov += bool(set(range(a, z + 1)) & mbins)
    rows.append(dict(pairs=name, pairs_in_tmaze=n_pairs, gc_fields=n_f, overlapping=n_ov, fraction=round(n_ov / n_f, 3) if n_f else np.nan))
print(pd.DataFrame(rows).to_string())

**Result: reproduced.** Of 19 place fields of GC with a detected MC partner on the T-maze (28 pairs), 5 overlap a field of the partner (0.263), within a third. On the authors' pairs (18 in T-maze files, as the paper's 18), 3 of 14 GC fields overlap.

## inheritance-openfield

> In the open fields, the rate maps of connected granule cells and mossy cells are not correlated within the granule cell's place field (median near 0, n = 19, Wilcoxon P = 0.60).

For detected GC to MC pairs, in each open-field session (A, B, A', the curtain and other-room sessions) where the GC has a place field, the correlation of the GC and MC maps over the field's pixels; Wilcoxon signed-rank test against 0.

In [ ]:
res_of = {}
Rp = {r["path"]: r for r in R}
for name, pairs_ in [("automatic test", sig), ("authors' table", va)]:
    cs = []
    for p in pairs_.itertuples():
        of = Rp[p.path].get("openfield") or {}
        for s, v in of.items():
            fl = v["field"].get(p.pre)
            if fl is None or p.post not in v["maps"]:
                continue
            mask = fl[0]
            a, b = v["maps"][p.pre][mask], v["maps"][p.post][mask]
            ok = np.isfinite(a) & np.isfinite(b)
            if ok.sum() > 2 and a[ok].std() > 0 and b[ok].std() > 0:
                cs.append(np.corrcoef(a[ok], b[ok])[0, 1])
    cs = np.array(cs)
    res_of[name] = cs
    if len(cs) >= 1:
        print(f"{name}: comparisons {len(cs)}, median r {np.median(cs):.3f}, Wilcoxon P = {stats.wilcoxon(cs).pvalue if len(cs) > 1 else np.nan:.3g}")
    else:
        print(f"{name}: no comparisons")

**Result: reproduced.** Over 18 comparisons the median correlation of GC and MC maps within the GC field is -0.005, Wilcoxon P = 0.25, so no correlation is found. On the authors' pairs, 14 comparisons, median -0.117, P = 0.90.

## gc-mc-distance and gc-backpropagation (blocked)

> Granule cells reside 59 +- 35 um above the mossy cells they drive.

> Granule cell spikes show a delay from the soma to the proximal dendrites, absent in mossy cells.

Both need the vertical position of each recording site on its shank. The electrodes table holds `channel_id`, `group_name`, `location`, `hilus_reference` and `shank_electrode_number` (the channel's place in the shank's list in the lab's .xml file), and no coordinate; the probe description says only that sites are staggered 20 um apart vertically. Which list position is which depth is not stated, so the distance between a GC site and its MC partner's site, and the order of sites along the somatodendritic axis, cannot be read. Both claims are blocked (`metadata_insufficient`); the claims file gives no criterion for either.

In [ ]:
f = S.open_asset([a for a in assets if "YutaMouse41-150903" in a["path"]][0]["asset_id"])
e = f["general/extracellular_ephys/electrodes"]
print("electrodes table columns:", list(e.keys()))
print("probe:", f["general/devices/probe"].attrs["description"])
f.close()

## Summary

| Claim | Outcome |
|---|---|
| unit-counts | reproduced |
| first-stage-labels | partially reproduced |
| interneuron-rates | reproduced |
| gc-narrower-waveform | reproduced |
| optotag-drd2 | partially reproduced |
| mc-more-active | reproduced |
| gc-nrem-highest | partially reproduced |
| ca3-wake-highest | not reproduced |
| nrem-wake-ratio | reproduced |
| theta-modulation | reproduced |
| gamma-modulation | reproduced |
| gamma-rate-correlation | not reproduced |
| ds2-recruitment | reproduced |
| monosyn-gc-to-mc | partially reproduced |
| burst-facilitation | reproduced |
| state-transmission | not reproduced |
| zero-lag-synchrony | partially reproduced |
| tmaze-central-active | partially reproduced |
| tmaze-multiple-fields | partially reproduced (rests on an assumption) |
| tmaze-peak-info | reproduced |
| tmaze-splitters | reproduced |
| remap-positive | reproduced |
| remap-gc-lt-mc | not reproduced |
| remap-fractions | partially reproduced |
| pv-remap | reproduced (seed sensitive) |
| inheritance-tmaze | reproduced (rests on an assumption) |
| inheritance-openfield | reproduced |
| mc-deeper | reproduced |
| gc-mc-classification | reproduced |
| gc-mc-distance | blocked: metadata insufficient |
| gc-backpropagation | blocked: metadata insufficient |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. Cell types are the authors' labels in the files, not re-derived. Monosynaptic and zero-lag pairs come from the Methods' automatic test without the paper's manual inspection; the authors' curated tables in the files are shown beside. Optogenetic suppression uses the Methods' 20% decrease with no significance test. On the T-maze the run starts at the mouse's last entry into the central arm (the files do not mark the gate opening), and fields are found from the highest local maximum down. Phase modulation uses up to an hour of `awake` time per file. DS2 amplitudes average up to 600 events per file. The wide-band site and waveform rest on up to four 52 s blocks of the wide band.

What was found about the dataset. The labels match the paper's counts closely (583 GC and 503 MC against 586 and 500; 248 dentate narrow-waveform interneurons, as the paper's 248). The authors' monosynaptic table holds the paper's 32 GC to MC pairs and no MC to GC pair, and its zero-lag table gives 22.1% of MC with a partner (paper 23%). The open-field files hold 246 GC, 182 MC and 317 CA3 units (paper 247, 181, 317). The seven same-day 'b' files are separate recordings, not duplicates. The stored-waveform site agrees with the wide-band site for 89% of GC and MC. The trials' `arm_start` lies well inside the central arm, not at its entry (T-maze section). The stored 1.6 ms waveforms end before the spike's peak for many units, so trough-to-peak latencies from them are lower bounds. No electrode coordinates are given, which blocks the two claims on site depth.